# TTS RTF & TTFA Benchmark

Đo 2 metrics cho TTS service (VieNeu-TTS):
- **RTF** (Real-Time Factor) = `synthesis_time / audio_duration` — càng nhỏ càng tốt
- **TTFA** (Time To First Audio) = thời gian đến chunk PCM đầu tiên — phản ánh perceived latency

**Yêu cầu**: TTS service đang chạy (`python tts/server.py` hoặc Docker)

**Input**: `evaluation/synthetic_qa.jsonl` — lấy câu trả lời dài từ field `answer`  
**Output**: `evaluation/tts_rtf_results/tts_rtf.jsonl` + `_summary.json`

```
RTF  < 1.0  → real-time capable
TTFA < 500ms → acceptable cho voice UX
```

In [ ]:
import json, time, asyncio, aiohttp
from pathlib import Path
import numpy as np

TTS_URL    = 'http://localhost:50053'
JSONL_PATH = Path('evaluation/synthetic_qa.jsonl')
OUT_DIR    = Path('evaluation/tts_rtf_results')
OUT_DIR.mkdir(exist_ok=True)

N_BENCH    = 200   # số câu benchmark
N_WARMUP   = 5
SAMPLE_RATE = 24_000

# Load queries — dùng field answer làm TTS input
records = []
with open(JSONL_PATH, encoding='utf-8') as f:
    for line in f:
        r = json.loads(line)
        if r.get('answer', '').strip():
            records.append(r)

print(f'Loaded {len(records)} records with answers')
lens = [len(r['answer']) for r in records]
print(f'Answer length — min={min(lens)}  mean={np.mean(lens):.0f}  max={max(lens)} chars')

In [ ]:
# Check TTS service
import urllib.request
try:
    with urllib.request.urlopen(f'{TTS_URL}/health', timeout=5) as r:
        print('TTS health:', json.loads(r.read()))
except Exception as e:
    print(f'TTS service not reachable: {e}')
    print('Start with: cd nutrition-callbot && python tts/server.py')

In [ ]:
# ── Hàm đo RTF qua /speak (non-streaming) ─────────────────────────────────
# TTS server trả về header X-TTFB-ms, X-RTF, X-Duration-s
# → dùng trực tiếp, không cần tự tính

async def measure_speak(session, text: str, idx: str):
    """POST /speak → đọc headers RTF + TTFB từ server."""
    t0 = time.perf_counter()
    async with session.post(
        f'{TTS_URL}/speak',
        json={'text': text, 'session_id': f'bench_{idx}'},
        timeout=aiohttp.ClientTimeout(total=120),
    ) as resp:
        resp.raise_for_status()
        _ = await resp.read()  # consume body
        total_ms = (time.perf_counter() - t0) * 1000

        ttfb_ms   = float(resp.headers.get('X-TTFB-ms', 0))
        rtf       = float(resp.headers.get('X-RTF', 0))
        duration_s = float(resp.headers.get('X-Duration-s', 0))

    return {
        'ttfb_ms':    round(ttfb_ms, 1),
        'rtf':        round(rtf, 4),
        'duration_s': round(duration_s, 3),
        'total_ms':   round(total_ms, 1),
    }


# ── Hàm đo TTFA qua /speak/stream (streaming) ─────────────────────────────
# TTFA = thời gian từ lúc gửi request đến khi nhận byte PCM đầu tiên

async def measure_stream(session, text: str, idx: str):
    """POST /speak/stream → đo TTFA và RTF từ stream."""
    t0 = time.perf_counter()
    ttfa_ms = None
    pcm_bytes = 0

    async with session.post(
        f'{TTS_URL}/speak/stream',
        json={'text': text, 'session_id': f'bench_stream_{idx}'},
        timeout=aiohttp.ClientTimeout(total=120),
    ) as resp:
        resp.raise_for_status()
        async for chunk in resp.content.iter_chunked(4096):
            if ttfa_ms is None:
                ttfa_ms = (time.perf_counter() - t0) * 1000
            pcm_bytes += len(chunk)

    total_s    = time.perf_counter() - t0
    duration_s = pcm_bytes / 2 / SAMPLE_RATE  # int16 = 2 bytes/sample
    rtf_stream = total_s / duration_s if duration_s > 0 else 0.0

    return {
        'ttfa_ms':    round(ttfa_ms or 0, 1),
        'rtf_stream': round(rtf_stream, 4),
        'duration_s': round(duration_s, 3),
        'total_ms':   round(total_s * 1000, 1),
    }

print('Functions defined.')

In [ ]:
# Warmup
print(f'Warming up ({N_WARMUP} samples)...')
async def warmup():
    async with aiohttp.ClientSession() as session:
        for r in records[:N_WARMUP]:
            await measure_speak(session, r['answer'][:300], r['id'])
            await measure_stream(session, r['answer'][:300], r['id'])
asyncio.run(warmup())
print('Done.')

In [ ]:
# Benchmark
bench = records[:N_BENCH]
results = []

async def run_benchmark():
    async with aiohttp.ClientSession() as session:
        for i, r in enumerate(bench):
            text = r['answer'].strip()

            speak  = await measure_speak(session,  text, r['id'])
            stream = await measure_stream(session, text, r['id'])

            results.append({
                'idx':        i,
                'id':         r['id'],
                'text_len':   len(text),
                # /speak metrics (server-side)
                'ttfb_ms':    speak['ttfb_ms'],
                'rtf':        speak['rtf'],
                'duration_s': speak['duration_s'],
                'total_ms':   speak['total_ms'],
                # /speak/stream metrics (client-side)
                'ttfa_ms':    stream['ttfa_ms'],
                'rtf_stream': stream['rtf_stream'],
            })

            if (i + 1) % 20 == 0:
                rtfs  = [r['rtf']     for r in results]
                ttfas = [r['ttfa_ms'] for r in results]
                print(f'[{i+1}/{len(bench)}]  '
                      f'RTF P50={np.median(rtfs):.3f}  P95={np.percentile(rtfs,95):.3f}  '
                      f'TTFA P50={np.median(ttfas):.0f}ms  P95={np.percentile(ttfas,95):.0f}ms')

asyncio.run(run_benchmark())
print('\nBenchmark complete.')

In [ ]:
# Save JSONL
out_jsonl = OUT_DIR / 'tts_rtf.jsonl'
with open(out_jsonl, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Saved: {out_jsonl}')

In [ ]:
# Summary
rtfs   = [r['rtf']        for r in results]
ttfas  = [r['ttfa_ms']    for r in results]
ttfbs  = [r['ttfb_ms']    for r in results]
totals = [r['total_ms']   for r in results]
durs   = [r['duration_s'] for r in results]

summary = {
    'model':     'vieneu_tts_0.3b_q4',
    'n_samples': len(results),
    'rtf': {
        'mean':   round(float(np.mean(rtfs)),           4),
        'median': round(float(np.median(rtfs)),         4),
        'p95':    round(float(np.percentile(rtfs, 95)), 4),
        'p99':    round(float(np.percentile(rtfs, 99)), 4),
    },
    'ttfa_ms': {
        'mean':   round(float(np.mean(ttfas)),           1),
        'median': round(float(np.median(ttfas)),         1),
        'p95':    round(float(np.percentile(ttfas, 95)), 1),
    },
    'ttfb_ms_server': {
        'mean':   round(float(np.mean(ttfbs)),           1),
        'median': round(float(np.median(ttfbs)),         1),
        'p95':    round(float(np.percentile(ttfbs, 95)), 1),
    },
    'total_ms': {
        'mean':   round(float(np.mean(totals)),           1),
        'median': round(float(np.median(totals)),         1),
        'p95':    round(float(np.percentile(totals, 95)), 1),
    },
    'audio_duration_s': {
        'mean':  round(float(np.mean(durs)),  2),
        'total': round(float(np.sum(durs)),   1),
    },
}

out_json = OUT_DIR / 'tts_rtf_summary.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('=' * 55)
print(f"Model   : {summary['model']}")
print(f"Samples : {summary['n_samples']}")
print(f"RTF     — median={summary['rtf']['median']}  P95={summary['rtf']['p95']}  P99={summary['rtf']['p99']}")
print(f"TTFA    — median={summary['ttfa_ms']['median']}ms  P95={summary['ttfa_ms']['p95']}ms")
print(f"TTFB(sv)— median={summary['ttfb_ms_server']['median']}ms  P95={summary['ttfb_ms_server']['p95']}ms")
print(f"Total   — median={summary['total_ms']['median']}ms  P95={summary['total_ms']['p95']}ms")
print('=' * 55)
print(f'Saved: {out_json}')

In [ ]:
# Plot
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 9))

def plot_hist(ax, data, label, unit, color='steelblue'):
    ax.hist(data, bins=40, color=color, edgecolor='black', alpha=0.75)
    ax.axvline(np.median(data),          color='red',    linestyle='--', lw=1.5,
               label=f'P50={np.median(data):.{2 if unit=="ms" else 3}f}')
    ax.axvline(np.percentile(data, 95),  color='orange', linestyle='--', lw=1.5,
               label=f'P95={np.percentile(data, 95):.{2 if unit=="ms" else 3}f}')
    ax.set_xlabel(f'{label} ({unit})')
    ax.set_ylabel('Count')
    ax.set_title(f'{label} Distribution')
    ax.legend()

plot_hist(axes[0,0], rtfs,   'RTF',          '',   'steelblue')
plot_hist(axes[0,1], ttfas,  'TTFA',         'ms', 'darkorange')
plot_hist(axes[1,0], ttfbs,  'TTFB (server)','ms', 'seagreen')
plot_hist(axes[1,1], totals, 'Total latency','ms', 'orchid')

plt.suptitle('VieNeu-TTS 0.3B Q4 — RTF & Latency', fontsize=13, fontweight='bold')
plt.tight_layout()
out_png = OUT_DIR / 'tts_rtf_dist.png'
plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')

In [ ]:
# Scatter: text length vs RTF và TTFA
text_lens = [r['text_len'] for r in results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(text_lens, rtfs,  alpha=0.4, s=15, color='steelblue')
axes[0].set_xlabel('Text length (chars)')
axes[0].set_ylabel('RTF')
axes[0].set_title('RTF vs Text Length')

axes[1].scatter(text_lens, ttfas, alpha=0.4, s=15, color='darkorange')
axes[1].set_xlabel('Text length (chars)')
axes[1].set_ylabel('TTFA (ms)')
axes[1].set_title('TTFA vs Text Length')

plt.tight_layout()
out_scatter = OUT_DIR / 'tts_rtf_scatter.png'
plt.savefig(str(out_scatter), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_scatter}')